Question 1
Consider a standard deck of 52 cards. What is the probability of drawing a red card (hearts or diamonds)? Now, given that you have drawn a red card, what is the probability that the card is a heart? Given that the drawn card is a face card (Jack, Queen, King), what is the probability that it is a diamond? Given that the card drawn is a face card, calculate the probability that it is also a spade or a queen.


In [ ]:
from fractions import Fraction
total = 52
red = 26
p_red = Fraction(red, total)
p_heart_given_red = Fraction(13, red)
face_cards = 12
p_diamond_given_face = Fraction(3, face_cards)
spade_faces = 3
queen_faces = 4
spade_or_queen_faces = spade_faces + queen_faces - 1
p_spade_or_queen_given_face = Fraction(spade_or_queen_faces, face_cards)
print('P(red) =', float(p_red), f'({p_red})')
print('P(heart | red) =', float(p_heart_given_red), f'({p_heart_given_red})')
print('P(diamond | face) =', float(p_diamond_given_face), f'({p_diamond_given_face})')
print('P(spade or queen | face) =', float(p_spade_or_queen_given_face), f'({p_spade_or_queen_given_face})')


Question 2
You are given a Bayesian Network structure to model student exam performance. Construct the network and CPTs, then compute the probability that the student passes the exam given StudyHours = Sufficient and Difficulty = Hard, and compute the probability that the student has High Intelligence given Pass = Yes.


In [ ]:
import itertools
priors = {
    'I': {'High': 0.7, 'Low': 0.3},
    'S': {'Sufficient': 0.6, 'Insufficient': 0.4},
    'D': {'Hard': 0.4, 'Easy': 0.6},
}
cpt_G = {
    ('High','Sufficient','Easy'): {'A': 0.8, 'B': 0.15, 'C': 0.05},
    ('High','Sufficient','Hard'): {'A': 0.6, 'B': 0.3, 'C': 0.1},
    ('High','Insufficient','Easy'): {'A': 0.5, 'B': 0.35, 'C': 0.15},
    ('High','Insufficient','Hard'): {'A': 0.3, 'B': 0.4, 'C': 0.3},
    ('Low','Sufficient','Easy'): {'A': 0.4, 'B': 0.4, 'C': 0.2},
    ('Low','Sufficient','Hard'): {'A': 0.2, 'B': 0.5, 'C': 0.3},
    ('Low','Insufficient','Easy'): {'A': 0.1, 'B': 0.5, 'C': 0.4},
    ('Low','Insufficient','Hard'): {'A': 0.05, 'B': 0.35, 'C': 0.6},
}
cpt_P = {'A': 0.95, 'B': 0.80, 'C': 0.50}
states = {
    'I': ['High', 'Low'],
    'S': ['Sufficient','Insufficient'],
    'D': ['Hard','Easy'],
    'G': ['A','B','C'],
}
joint = []
for I,S,D,G in itertools.product(states['I'], states['S'], states['D'], states['G']):
    p = priors['I'][I] * priors['S'][S] * priors['D'][D] * cpt_G[(I,S,D)][G] * cpt_P[G]
    joint.append(({'I': I, 'S': S, 'D': D, 'G': G, 'P': 'Yes'}, p))
    joint.append(({'I': I, 'S': S, 'D': D, 'G': G, 'P': 'No'}, p * (1 - cpt_P[G])))
def query(variable, evidence):
    numerator = 0.0
    denominator = 0.0
    for inst, p in joint:
        if all(inst[k] == v for k,v in evidence.items()):
            denominator += p
            if inst[variable] == evidence.get(variable, inst[variable]):
                numerator += p
    return numerator / denominator if denominator else 0.0
p_pass = query('P', {'S': 'Sufficient', 'D': 'Hard', 'P': 'Yes'})
p_int_high = query('I', {'P': 'Yes', 'I': 'High'})
print('P(Pass=Yes | StudyHours=Sufficient, Difficulty=Hard) =', round(p_pass, 4))
print('P(Intelligence=High | Pass=Yes) =', round(p_int_high, 4))


Question 3
Build a Bayesian Network with Disease influencing Fever, Cough, Fatigue, and Chills. Use the CPTs below and compute the posterior probability of the disease given symptoms, then compute P(Fatigue=Yes | Disease=Flu).


In [ ]:
p_disease = {'Flu': 0.3, 'Cold': 0.7}
p_fever = {'Flu': {'Yes': 0.9, 'No': 0.1}, 'Cold': {'Yes': 0.5, 'No': 0.5}}
p_cough = {'Flu': {'Yes': 0.8, 'No': 0.2}, 'Cold': {'Yes': 0.6, 'No': 0.4}}
p_fatigue = {'Flu': {'Yes': 0.7, 'No': 0.3}, 'Cold': {'Yes': 0.3, 'No': 0.7}}
p_chills = {'Flu': {'Yes': 0.6, 'No': 0.4}, 'Cold': {'Yes': 0.4, 'No': 0.6}}
def posterior(evidence):
    scores = {}
    total = 0.0
    for disease in p_disease:
        p = p_disease[disease] * p_fever[disease][evidence['Fever']] * p_cough[disease][evidence['Cough']]
        if 'Chills' in evidence:
            p *= p_chills[disease][evidence['Chills']]
        scores[disease] = p
        total += p
    return {d: p / total for d,p in scores.items()} if total else {d: 0.0 for d in scores}
posterior_1 = posterior({'Fever': 'Yes', 'Cough': 'Yes'})
posterior_2 = posterior({'Fever': 'Yes', 'Cough': 'Yes', 'Chills': 'Yes'})
p_fatigue_given_flu = p_fatigue['Flu']['Yes']
print('P(Disease | Fever=Yes, Cough=Yes) =', posterior_1)
print('P(Disease | Fever=Yes, Cough=Yes, Chills=Yes) =', posterior_2)
print('P(Fatigue=Yes | Disease=Flu) =', p_fatigue_given_flu)


Question 4
Define a simple Markov Model with weather states Sunny, Cloudy, and Rainy. Create a transition matrix, simulate weather for the next 10 days starting from Sunny, and calculate the probability of at least 3 rainy days in the 10-day period.


In [ ]:
trans = {
    'Sunny': {'Sunny': 0.6, 'Cloudy': 0.3, 'Rainy': 0.1},
    'Cloudy': {'Sunny': 0.3, 'Cloudy': 0.4, 'Rainy': 0.3},
    'Rainy': {'Sunny': 0.2, 'Cloudy': 0.3, 'Rainy': 0.5},
}
days = 10
dp = {('Sunny', 0): 1.0}
for _ in range(1, days):
    new_dp = {}
    for (state, count), prob in dp.items():
        for nxt, p in trans[state].items():
            key = (nxt, count + (1 if nxt == 'Rainy' else 0))
            new_dp[key] = new_dp.get(key, 0.0) + prob * p
    dp = new_dp
prob_at_least_3 = sum(p for (state, count), p in dp.items() if count >= 3)
print('Probability of at least 3 rainy days in the 10-day period =', round(prob_at_least_3, 4))
